# PCAP → CSV Feature Extraction Pipeline

## Purpose

This notebook is the **data preprocessing pipeline** for the tIDS (Traffic-based Intrusion Detection System) project.  
It reads raw network captures (`.pcap` / `.pcapng`), extracts **38 behavioral features** per 1-second time window, and saves everything into a single labeled CSV file (`tIDS_final_data.csv`) ready to train a Machine Learning classifier.

---

## Input Files

8 PCAP files are processed, each representing a different traffic scenario:

| File | Label | Attack Category |
|------|-------|----------------|
| `Normal.pcap` | `Normal` | Baseline legitimate traffic |
| `SYN_Flood.pcap` | `SYN_Flood` | DDoS / Flood |
| `UDP_Flood.pcap` | `UDP_Flood` | DDoS / Flood |
| `Smurf.pcap` | `Smurf` | DDoS / Flood (ICMP amplification) |
| `ARP_Spoof.pcapng` | `ARP_Spoof` | Spoofing |
| `DNS_Spoof.pcapng` | `DNS_Spoof` | Spoofing |
| `STP_Spoof.pcapng` | `STP_Spoof` | Spoofing |
| `IGMP_Spoof.pcap` | `IGMP_Spoof` | Spoofing |

---

## Pipeline Steps

### Step 1 — Streaming with `PcapReader`

Instead of loading the entire file into memory at once (which would crash on multi-GB captures), Scapy's `PcapReader` **streams packets one by one**. This keeps RAM usage constant regardless of file size.

---

### Step 2 — 1-Second Time Window Aggregation

Every packet is assigned to a **1-second bucket** based on its timestamp. Each bucket is stored in a `defaultdict` keyed by `ts`. At the end, each bucket becomes **one row** in the CSV.  
This approach captures *behavioral patterns* (rates, ratios, diversity) rather than raw per-packet values — exactly what ML models need.

---

### Step 3 — Per-Packet Feature Counting

For every packet, the code inspects each protocol layer and increments the relevant counters in the current window:

#### 🔹 Volume Metrics

Tracks raw traffic intensity. High `packet_rate` or `byte_rate` signals flooding. Small `mean_pkt_size` is typical of SYN/ICMP floods (tiny header-only packets).

#### 🔹 Broadcast & Multicast Detection

Detection happens at **both** Layer 2 (MAC address) and Layer 3 (IP address) to catch all broadcast/multicast variants. High `broadcast_pkt_count` is a strong Smurf attack signal.

#### 🔹 TCP Flag Analysis

Flag counts are kept **pure** — a SYN-ACK packet is counted as neither a SYN nor an ACK. This ensures `syn_to_ack_ratio` accurately reflects one-sided SYN flooding: in a SYN flood, attackers send thousands of SYNs but never complete the handshake, so ACKs stay near zero and the ratio spikes.

#### 🔹 UDP & ICMP Counting
Simple packet counts plus ICMP type discrimination:
- `type == 8` → Echo Request (ping sent)  
- `type == 0` → Echo Reply (ping response)  

In a **Smurf attack**, an attacker sends a single ICMP request to a broadcast address, causing all hosts on the network to reply to the victim — so `icmp_reply_to_request_ratio` becomes much greater than 1.

#### 🔹 ARP Spoof Detection

A dictionary `ip_mac_map` (reset for each file) records the **first MAC address seen** for every source IP. If a later ARP packet claims the same IP but with a *different* MAC, it's a conflict — the definitive signature of ARP cache poisoning. `ip_mac_mapping_changes > 0` is a near-certain spoof indicator.

Gratuitous ARP (where `psrc == pdst`) is also counted separately — legitimate hosts rarely send these, so a high count is suspicious.

#### 🔹 DNS Analysis

In normal traffic, queries and responses are roughly balanced. In DNS spoofing, an attacker injects **fake responses** — so `dns_reply_to_query_ratio` rises above 1. Spoofed responses also tend to use **abnormally low TTLs** to limit how long victims cache the fake record.

#### 🔹 STP Detection (with LLC/802.3 Fallback)

STP packets use raw **802.3 Ethernet + LLC encapsulation** (not standard IP), so Scapy sometimes fails to parse them as `STP`. The fallback manually inspects the LLC payload bytes for the STP protocol identifier (`0x0000`). This two-stage detection ensures no STP packet is missed.

#### 🔹 IGMP Analysis

IGMP controls multicast group membership. Types `0x16` (v2 Join) and `0x22` (v3 Report) indicate a host joining a multicast group. A flood of fake join messages is the signature of IGMP spoofing.

---

### Step 4 — Derived Features & Safe Ratios

After counting, behavioral ratios are computed. `safe_ratio` prevents `ZeroDivisionError` when a denominator is zero. Two critical ratios are also **clipped** to prevent extreme outliers from destabilizing the ML model:
- `syn_to_ack_ratio` → capped at `10.0`
- `dns_reply_to_query_ratio` → capped at `5.0`

---

### Step 5 — CSV Output
Each completed window is written as one row with all 38 features + the attack label. Windows are written in **chronological order** (`sorted(windows.keys())`).

---

## The 38 Features at a Glance

| # | Feature | Attack Signal |
|---|---------|--------------|
| 1–4 | `packet_rate`, `byte_rate`, `mean_pkt_size`, `max_pkt_size` | High rate / small size → flood |
| 5–11 | TCP flag counts + ratios | `syn_to_ack >> 1` → SYN flood |
| 12–18 | UDP/ICMP counts + echo ratio | `icmp_reply > request` → Smurf |
| 19–24 | ARP counts + `ip_mac_mapping_changes` | Changes > 0 → ARP spoof |
| 25–29 | DNS counts + `dns_ttl_mean` | Low TTL + extra replies → DNS spoof |
| 30–31 | STP count + topology changes | TCN flood → STP spoof |
| 32–33 | IGMP count + join events | Mass joins → IGMP spoof |
| 34–38 | Unique IPs, ports, broadcast/multicast | High diversity → DDoS / scan |

---

##  Output: `tIDS_final_data.csv`

- **Rows**: one per 1-second time window  
- **Columns**: 38 numerical features + `label`  
- **Labels**: `Normal`, `SYN_Flood`, `UDP_Flood`, `Smurf`, `ARP_Spoof`, `DNS_Spoof`, `STP_Spoof`, `IGMP_Spoof`

In [ ]:
from scapy.all import PcapReader, TCP, UDP, ICMP, ARP, DNS, STP, Ether
from scapy.contrib.igmp import IGMP
from scapy.layers.l2 import Dot3, LLC
import csv, time
from collections import defaultdict

# ─── Attack PCAP files ───
Attacks = {
    "IGMP_Spoof.pcap"  : "IGMP_Spoof",
    "STP_Spoof.pcapng" : "STP_Spoof",
    "UDP_Flood.pcap"   : "UDP_Flood",
    "DNS_Spoof.pcapng" : "DNS_Spoof",
    "Normal.pcap"      : "Normal",
    "SYN_Flood.pcap"   : "SYN_Flood",
    "ARP_Spoof.pcapng" : "ARP_Spoof",
    "Smurf.pcap"       : "Smurf",
}

# ─── Safe division helper ───
def safe_ratio(a, b):
    return round(a / b, 4) if b > 0 else 0.0

# ─── CSV column headers (38 features + label) ───
COLUMNS = [
    "packet_rate", "byte_rate", "mean_pkt_size", "max_pkt_size",
    "tcp_syn_count", "tcp_ack_count", "tcp_rst_count", "tcp_fin_count",
    "syn_to_ack_ratio", "rst_to_syn_ratio", "tcp_to_total_ratio",
    "udp_packet_count", "udp_to_total_ratio",
    "icmp_packet_count", "icmp_to_total_ratio",
    "icmp_echo_request_count", "icmp_echo_reply_count", "icmp_reply_to_request_ratio",
    "arp_request_count", "arp_reply_count", "arp_reply_ratio",
    "gratuitous_arp_count", "gratuitous_to_reply_ratio", "ip_mac_mapping_changes",
    "dns_query_count", "dns_response_count", "dns_reply_to_query_ratio",
    "dns_avg_response_size", "dns_ttl_mean",
    "stp_packet_count", "stp_topology_change",
    "igmp_packet_count", "igmp_join_count",
    "unique_src_ip_count", "unique_dst_ip_count", "unique_dst_ports",
    "broadcast_pkt_count", "multicast_pkt_count",
    "label",
]

# ─── Default window template ───
def new_window():
    return {
        "packet_count": 0, "total_bytes": 0, "pkt_sizes": [],
        "tcp_pkt_count": 0, "tcp_syn": 0, "tcp_ack": 0, "tcp_rst": 0, "tcp_fin": 0,
        "udp_pkt": 0,
        "icmp_pkt": 0, "icmp_echo_req": 0, "icmp_echo_rep": 0,
        "arp_req": 0, "arp_rep": 0, "grat_arp": 0, "ip_mac_changes": 0,
        "dns_qry": 0, "dns_resp": 0, "dns_resp_sizes": [], "dns_ttls": [],
        "stp_pkt": 0, "stp_tc": 0,
        "igmp_pkt": 0, "igmp_join": 0,
        "src_ips": set(), "dst_ips": set(), "dst_ports": set(),
        "bcast": 0, "mcast": 0,
    }

# ─── Main processing ───
csv_out = "tIDS_final_data.csv"

with open(csv_out, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(COLUMNS)

    for file_name, label in Attacks.items():
        print(f"{'='*50}")
        print(f"Processing: {file_name} → {label}")

        start_time = time.time()
        windows    = defaultdict(new_window)
        ip_mac_map = {}
        i = 0

        with PcapReader(file_name) as reader:
            for pkt in reader:
                try:
                    ts = int(float(pkt.time) // 1) * 1
                    w  = windows[ts]

                    pkt_len = len(pkt)
                    w["packet_count"] += 1
                    w["total_bytes"]  += pkt_len
                    w["pkt_sizes"].append(pkt_len)

                    # ── Broadcast / Multicast ──
                    is_bcast, is_mcast = False, False
                    if pkt.haslayer(Ether):
                        dst_mac = pkt[Ether].dst.lower()
                        if dst_mac == "ff:ff:ff:ff:ff:ff":
                            is_bcast = True
                        elif dst_mac.startswith("01:00:5e") or dst_mac.startswith("33:33"):
                            is_mcast = True

                    if pkt.haslayer("IP"):
                        src_ip = pkt["IP"].src
                        dst_ip = pkt["IP"].dst
                        w["src_ips"].add(src_ip)
                        w["dst_ips"].add(dst_ip)
                        if dst_ip.endswith(".255") or dst_ip == "255.255.255.255":
                            is_bcast = True
                        if 224 <= int(dst_ip.split(".")[0]) <= 239:
                            is_mcast = True

                    if is_bcast: w["bcast"] += 1
                    if is_mcast: w["mcast"] += 1

                    # ── TCP ──
                    if TCP in pkt:
                        w["tcp_pkt_count"] += 1
                        flags = pkt[TCP].flags
                        if (flags & 0x02) and not (flags & 0x10): w["tcp_syn"] += 1
                        if (flags & 0x10) and not (flags & 0x02) and not (flags & 0x04) and not (flags & 0x01):
                            w["tcp_ack"] += 1
                        if flags & 0x04: w["tcp_rst"] += 1
                        if flags & 0x01: w["tcp_fin"] += 1
                        w["dst_ports"].add(pkt[TCP].dport)

                    # ── UDP ──
                    if UDP in pkt:
                        w["udp_pkt"] += 1
                        w["dst_ports"].add(pkt[UDP].dport)

                    # ── ICMP ──
                    if ICMP in pkt:
                        w["icmp_pkt"] += 1
                        t = pkt[ICMP].type
                        if t == 8:  w["icmp_echo_req"] += 1
                        elif t == 0: w["icmp_echo_rep"] += 1

                    # ── ARP ──
                    if ARP in pkt:
                        arp = pkt[ARP]
                        if arp.op == 1: w["arp_req"] += 1
                        elif arp.op == 2: w["arp_rep"] += 1
                        if arp.psrc == arp.pdst or arp.pdst == "0.0.0.0":
                            w["grat_arp"] += 1
                        sip, smac = arp.psrc, arp.hwsrc
                        if sip in ip_mac_map:
                            if smac != ip_mac_map[sip]:
                                w["ip_mac_changes"] += 1
                        else:
                            ip_mac_map[sip] = smac

                    # ── DNS ──
                    if DNS in pkt:
                        dns = pkt[DNS]
                        if dns.qr == 0:
                            w["dns_qry"] += 1
                        else:
                            w["dns_resp"] += 1
                            w["dns_resp_sizes"].append(len(dns))
                            try:
                                if dns.ancount and dns.an:
                                    w["dns_ttls"].append(dns.an.ttl)
                            except Exception:
                                pass

                    # ── STP (+ LLC/802.3 fallback) ──
                    if pkt.haslayer(STP):
                        w["stp_pkt"] += 1
                        try:
                            if pkt[STP].bpduflags & 0x01: w["stp_tc"] += 1
                        except Exception:
                            pass
                    elif pkt.haslayer(Dot3) and pkt.haslayer(LLC):
                        try:
                            payload = bytes(pkt[LLC].payload)
                            if len(payload) >= 4 and payload[0:2] == b'\x00\x00':
                                w["stp_pkt"] += 1
                                if payload[3] == 0x00 and len(payload) > 4 and payload[4] & 0x01:
                                    w["stp_tc"] += 1
                                elif payload[3] == 0x80:
                                    w["stp_tc"] += 1
                        except Exception:
                            pass

                    # ── IGMP ──
                    if pkt.haslayer(IGMP):
                        w["igmp_pkt"] += 1
                        try:
                            if pkt[IGMP].type in [0x12, 0x16, 0x22]:
                                w["igmp_join"] += 1
                        except Exception:
                            pass

                    i += 1
                    if i % 100000 == 0:
                        print(f"  ... {i:,} packets ({time.time()-start_time:.1f}s)")

                except Exception:
                    i += 1
                    continue

        # ── Write derived features per window ──
        for ts in sorted(windows.keys()):
            d  = windows[ts]
            pr = d["packet_count"]
            br = d["total_bytes"]
            writer.writerow([
                pr, br,
                round(br/pr, 2) if pr > 0 else 0,
                max(d["pkt_sizes"]) if d["pkt_sizes"] else 0,
                d["tcp_syn"], d["tcp_ack"], d["tcp_rst"], d["tcp_fin"],
                min(10.0, safe_ratio(d["tcp_syn"], d["tcp_ack"])),
                safe_ratio(d["tcp_rst"],  d["tcp_syn"]),
                safe_ratio(d["tcp_pkt_count"], pr),
                d["udp_pkt"], safe_ratio(d["udp_pkt"], pr),
                d["icmp_pkt"], safe_ratio(d["icmp_pkt"], pr),
                d["icmp_echo_req"], d["icmp_echo_rep"],
                safe_ratio(d["icmp_echo_rep"], d["icmp_echo_req"]),
                d["arp_req"], d["arp_rep"], safe_ratio(d["arp_rep"], pr),
                d["grat_arp"], safe_ratio(d["grat_arp"], d["arp_rep"]),
                d["ip_mac_changes"],
                d["dns_qry"], d["dns_resp"],
                min(5.0, safe_ratio(d["dns_resp"], d["dns_qry"])),
                round(sum(d["dns_resp_sizes"])/len(d["dns_resp_sizes"]),2) if d["dns_resp_sizes"] else 0,
                round(sum(d["dns_ttls"])/len(d["dns_ttls"]),2) if d["dns_ttls"] else 0,
                d["stp_pkt"], d["stp_tc"],
                d["igmp_pkt"], d["igmp_join"],
                len(d["src_ips"]), len(d["dst_ips"]), len(d["dst_ports"]),
                d["bcast"], d["mcast"],
                label,
            ])

        print(f"✓ {file_name} → {len(windows)} windows, {i:,} packets in {time.time()-start_time:.1f}s")

print(f"\n{'='*50}")
print(f" Dataset saved: {csv_out}")